# Train the “Hey Molty” wake word

This notebook trains Molty's wake-word classifier with LiveKit's `livekit-wakeword` pipeline. Training stays in Google Colab; the Mac and Raspberry Pi only run the exported ONNX model.

Before running:

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Run every cell in order.
3. Keep this tab open. The production configuration downloads roughly 18 GB and can take several hours.

The final cell downloads `hey_molty_colab_bundle.zip`, containing the ONNX model, evaluation results, recommended threshold, and training configuration.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No GPU detected. Select Runtime → Change runtime type → T4 GPU, "
        "then reconnect."
    )

subprocess.run(["nvidia-smi"], check=True)

## Install the LiveKit trainer

The explicit `numba` constraint avoids an incompatible legacy dependency selected by some Python resolvers.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq espeak-ng ffmpeg sox libsndfile1 portaudio19-dev
%pip install -q "numba>=0.60" "livekit-wakeword[train,eval,export]==0.2.1"

subprocess.run(["livekit-wakeword", "--help"], check=True)

## Configure “Hey Molty”

Phonetically similar phrases are explicit negative examples so that “Hey Molly,” “Hey Moldy,” and “Hey Multi” do not wake the robot.

In [ ]:
from pathlib import Path

import yaml

WORK_DIR = Path("/content/molty-wakeword")
WORK_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "model_name": "hey_molty",
    "target_phrases": ["hey molty"],
    "n_samples": 25000,
    "n_samples_val": 5000,
    "n_background_samples": 2000,
    "n_background_samples_val": 500,
    "tts_batch_size": 50,
    "custom_negative_phrases": [
        "molty",
        "hey molly",
        "hey moldy",
        "hey multi",
        "hey malty",
        "hey monty",
        "okay molty",
        "they called me",
        "mainly",
    ],
    "noise_scales": [0.98],
    "noise_scale_ws": [0.98],
    "length_scales": [0.75, 1.0, 1.25],
    "slerp_weights": [0.2, 0.35, 0.5, 0.65, 0.8],
    "data_dir": str(WORK_DIR / "data"),
    "output_dir": str(WORK_DIR / "output"),
    "augmentation": {
        "clip_duration": 2.0,
        "batch_size": 16,
        "rounds": 3,
        "background_paths": [str(WORK_DIR / "data" / "backgrounds")],
        "rir_paths": [str(WORK_DIR / "data" / "rirs")],
    },
    "model": {
        "model_type": "conv_attention",
        "model_size": "small",
    },
    "steps": 100000,
    "learning_rate": 0.0001,
    "weight_decay": 0.01,
    "label_smoothing": 0.05,
    "max_negative_weight": 3000,
    "target_fp_per_hour": 0.1,
    "batch_n_per_class": {
        "positive": 50,
        "adversarial_negative": 50,
        "ACAV100M_sample": 1024,
        "background_noise": 50,
    },
}

CONFIG_PATH = WORK_DIR / "hey_molty.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG, sort_keys=False), encoding="utf-8")
print(CONFIG_PATH.read_text(encoding="utf-8"))

## Download training data

This is the largest download. Re-running the cell in the same Colab session reuses files already present.

In [ ]:
subprocess.run(
    ["livekit-wakeword", "setup", "--config", str(CONFIG_PATH)],
    check=True,
)

## Train, export, and evaluate

LiveKit synthesizes positive and adversarial samples, adds noise and reverberation, extracts features, trains the classifier, exports ONNX, and measures false positives and recall.

In [ ]:
subprocess.run(
    ["livekit-wakeword", "run", str(CONFIG_PATH)],
    check=True,
)

## Review the result

In [ ]:
import json

from IPython import display as ipython_display

MODEL_DIR = Path(CONFIG["output_dir"]) / CONFIG["model_name"]
MODEL_PATH = MODEL_DIR / "hey_molty.onnx"
METRICS_PATH = MODEL_DIR / "hey_molty_eval.json"
DET_PATH = MODEL_DIR / "hey_molty_det.png"

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2))
print(f"\nRecommended MOLTY_WAKE_THRESHOLD={metrics['optimal_threshold']}")
ipython_display.display(ipython_display.Image(filename=str(DET_PATH)))

## Download the deployment bundle

Your browser downloads one ZIP containing `hey_molty.onnx`, the evaluation artifacts, the exact configuration, and an environment snippet with the recommended threshold.

In [ ]:
import zipfile

from google.colab import files

ENV_PATH = WORK_DIR / "molty-wakeword.env"
ENV_PATH.write_text(
    "MOLTY_WAKEWORD_MODEL=models/hey_molty.onnx\n"
    f"MOLTY_WAKE_THRESHOLD={metrics['optimal_threshold']}\n",
    encoding="utf-8",
)

BUNDLE_PATH = WORK_DIR / "hey_molty_colab_bundle.zip"
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(MODEL_PATH, arcname="hey_molty.onnx")
    bundle.write(METRICS_PATH, arcname="hey_molty_eval.json")
    bundle.write(DET_PATH, arcname="hey_molty_det.png")
    bundle.write(CONFIG_PATH, arcname="hey_molty.yaml")
    bundle.write(ENV_PATH, arcname="molty-wakeword.env")

print(f"Downloading {BUNDLE_PATH.name} ({BUNDLE_PATH.stat().st_size / 1024:.1f} KiB)")
files.download(str(BUNDLE_PATH))